# Photothermal PINN — Revision 2

In [ ]:
import os, io, functools, hashlib
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

from network import NetDiscovery, np_to_th, DEVICE
from diff_equations import grad

# a trained PINN is saved in cache/ and loaded the next time (same data and settings); set USE_CACHE = False to train again
USE_CACHE = True
CACHE_VERSION = 1
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)


def cache_file(name, *parts):
    h = hashlib.md5(str(CACHE_VERSION).encode())
    for p in parts:
        h.update(np.asarray(p, dtype=float).tobytes() if isinstance(p, np.ndarray) else repr(p).encode())
    return os.path.join(CACHE_DIR, f"{name}_{h.hexdigest()}.pt")


Tenv_true, a_true, tau_true = 25.0, 0.05, 300.0        # synthetic data: ambient temperature, heating rate, time constant


def heating_cooling(t, a, tau, t_off, Tenv=25.0):
    theta_off = a * tau * (1 - np.exp(-t_off / tau))
    return Tenv + np.where(t < t_off, a * tau * (1 - np.exp(-t / tau)), theta_off * np.exp(-(t - t_off) / tau))


def make_experiment(t_off_e, t_end, dt=10, noise=0.1, seed=0):
    rng = np.random.default_rng(seed)
    t_e = np.arange(0, t_end + dt / 2, dt)
    T_e = functools.partial(heating_cooling, a=a_true, tau=tau_true, t_off=t_off_e, Tenv=Tenv_true)(t_e)
    return t_e, T_e + rng.normal(0, noise, len(t_e))

EXPERIMENTS = {
    "full  (heat 5τ, cool 5τ)":  dict(t_off_e=5 * tau_true, t_end=10 * tau_true),
    "short (heat 1τ, cool 2τ)": dict(t_off_e=1 * tau_true, t_end= 3 * tau_true),
}

REAL_DATASETS = {      # file, optional nanoparticle name, laser switch-off time and ambient temperature of each curve
    "DS1":  dict(file="real_data/gnr_10x38_20ugml_NIR_laser.csv", t_off=900.0, Tenv=25.0),
    "DS2a": dict(file="real_data/gnr_10x41_peg_bare_808nm_2W.csv", nanoparticle="B-GNR_10x41nm_808nm_2W", t_off=1799.2, Tenv=25.47),
    "DS2b": dict(file="real_data/gnr_10x41_peg_bare_808nm_2W.csv", nanoparticle="PEG-GNR_10x41nm_808nm_2W", t_off=1799.2, Tenv=25.34),
    "DS3":  dict(file="real_data/aunr_785nm_1p2Wcm2_pmc13028931.csv", t_off=1347.4, Tenv=23.85),
    "DS4":  dict(file="real_data/aunrsi_785nm_1p2Wcm2_pmc13028931.csv", t_off=1255.1, Tenv=24.15),
}
REAL_BY_LABEL = REAL_DATASETS
DIAG_CURVE = "DS2b"


def load_real(ds):
    df = pd.read_csv(ds["file"])
    if "nanoparticle" in ds:
        df = df[df["nanoparticle"] == ds["nanoparticle"]]
    return df["time_s"].values.astype(float), df["T_C"].values.astype(float)

def simulate_loss(t, a_, f, t_off_):
    # numerical solution of dθ/dt = a L(t) - f(θ), θ(0) = 0
    rhs = lambda tt, y: [a_ * (tt < t_off_) - f(max(y[0], 0.0))]
    return solve_ivp(rhs, (0, t.max()), [0.0], t_eval=t, max_step=5.0, rtol=1e-8, atol=1e-10).y[0]


def single_tau_fit(t, th, t_off_):
    res = least_squares(lambda p: simulate_loss(t, p[0], lambda x: p[1] * x, t_off_) - th, [0.05, 3e-3],
                        bounds=([1e-5, 1e-6], [5, 1]), diff_step=1e-3, x_scale="jac")
    return res.x, np.sqrt(np.mean(res.fun ** 2))

In [ ]:
class PhotothermalPINN(NetDiscovery):
    """PINN for discovering (a, τ) from photothermal heating–cooling data."""

    def __init__(self, t_off_hat, switch_input=True,
                 n_units=100, epochs=3000, data_loss=nn.MSELoss(),
                 lr=3e-3, physics_loss=None, data_weight=1.0, physics_weight=1.0):
        n_in = 2 if switch_input else 1
        super().__init__(input_dim=n_in, output_dim=1,
                         n_units=n_units, epochs=epochs,
                         data_loss=data_loss, lr=lr,
                         physics_loss=physics_loss,
                         data_weight=data_weight, physics_weight=physics_weight)

        # Replace ReLU activations with tanh for smooth ODE gradients
        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.ReLU):
                self.layers[i] = nn.Tanh()

        self.t_off_hat    = t_off_hat
        self.switch_input = switch_input
        self.a            = nn.Parameter(data=torch.tensor([0.]))

    def forward(self, t):
        if self.switch_input:
            t = torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1)
        return self.out(self.layers(t))


def train_photothermal(t_, T_, t_off_, Tenv_=Tenv_true,
                       switch_input=True, data_weight=1.0, physics_weight=1.0,
                       adam_epochs=3000, lbfgs_iters=300, seed=0, return_fit=False, log=None):
    """Train PhotothermalPINN and return (a_est, tau_est), plus the network's fitted T if return_fit."""
    torch.manual_seed(seed)

    # Normalise inputs to [0, 1]
    t_scale = float(t_.max())
    T_scale = float(max((T_ - Tenv_).max(), 1e-3))
    t_hat, theta_hat = t_ / t_scale, (T_ - Tenv_) / T_scale
    t_off_hat    = t_off_ / t_scale

    def physics_loss(model):
        t_col = torch.linspace(0, 1, 800).view(-1, 1).requires_grad_(True).to(DEVICE)
        laser = (t_col < t_off_hat).float()
        temps = model(t_col)
        dT    = grad(temps, t_col)[0]
        pde   = model.a * laser - model.kappa * temps - dT
        ic    = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return torch.mean(pde ** 2) + ic

    model = PhotothermalPINN(t_off_hat, switch_input=switch_input,
                              physics_loss=physics_loss,
                              data_weight=data_weight, physics_weight=physics_weight,
                              epochs=adam_epochs, lr=3e-3).to(DEVICE)   # same device as the data
    Xt_ = np_to_th(t_hat.reshape(-1, 1))
    yt_ = np_to_th(theta_hat.reshape(-1, 1))
    cache_path = cache_file("photothermal", t_, T_, t_off_, Tenv_, switch_input, data_weight, physics_weight, adam_epochs, lbfgs_iters, seed)
    if USE_CACHE and log is None and os.path.exists(cache_path):
        model.load_state_dict(torch.load(cache_path, map_location=DEVICE))      # trained before: no training
    else:
        if log is not None:
            log.start(model, T_scale / t_scale)
        model.fit(t_hat.reshape(-1, 1), theta_hat.reshape(-1, 1),
                  on_step=None if log is None else lambda ep, d, p: log.record("Adam", d, p))
        if log is not None:
            log.end_adam()

        # L-BFGS fine-tuning
        opt = torch.optim.LBFGS(model.parameters(), lr=1.0,
                                  max_iter=lbfgs_iters, line_search_fn='strong_wolfe')
        def _closure():
            opt.zero_grad()
            data_term = model.data_loss(yt_, model(Xt_))
            physics_term = physics_loss(model)
            l = data_weight * data_term + physics_weight * physics_term
            l.backward()
            if log is not None:
                log.record("L-BFGS", data_term, physics_term)
            return l
        opt.step(_closure)
        if log is not None:
            log.finish(opt)
        if USE_CACHE and log is None:
            torch.save(model.state_dict(), cache_path)

    a_est, tau_est = model.a.item() * T_scale / t_scale, t_scale / model.kappa.item()
    if return_fit:   # the network's own output at the data times, in °C
        fit = model(Xt_).detach().cpu().numpy().ravel() * T_scale + Tenv_
        return a_est, tau_est, fit
    return a_est, tau_est


class LossNet(nn.Module):
    # f(θ) = θ·softplus(h(θ)): zero at ambient temperature, positive above it
    def __init__(self, n=20):
        super().__init__()
        self.h = nn.Sequential(nn.Linear(1, n), nn.Tanh(), nn.Linear(n, n), nn.Tanh(), nn.Linear(n, 1))

    def forward(self, th):
        return th * nn.functional.softplus(self.h(th))


class LossPINN(nn.Module):
    # temperature network with the switch input of Section 5, plus the learned loss law and the heating rate a
    def __init__(self, t_off_hat, n_units=100):
        super().__init__()
        self.t_off_hat = t_off_hat
        layers, d = [], 2
        for _ in range(4):
            layers += [nn.Linear(d, n_units), nn.Tanh()]
            d = n_units
        self.body = nn.Sequential(*layers, nn.Linear(n_units, 1))
        self.loss_law = LossNet()
        self.a = nn.Parameter(torch.tensor([1.0]))

    def forward(self, t):
        return self.body(torch.cat([t, torch.relu(t - self.t_off_hat)], dim=1))


def train_loss_pinn(t, th, t_off_, adam_epochs=8000, lbfgs_iters=1000, data_weight=1.0, physics_weight=1.0, seed=0, n_units=100, return_terms=False, log=None):
    torch.manual_seed(seed)
    t_scale, T_scale = float(t.max()), float(th.max())
    t_hat = np_to_th(t / t_scale); theta_hat = np_to_th(th / T_scale)
    model = LossPINN(t_off_ / t_scale, n_units=n_units).to(DEVICE)
    t_col = torch.linspace(0, 1, 800).view(-1, 1).to(DEVICE).requires_grad_(True)
    laser = (t_col < t_off_ / t_scale).float()

    def losses():                      # data loss and physics loss (ODE residual + initial condition)
        th_c = model(t_col)
        dth = grad(th_c, t_col)[0]
        res = dth - model.a * laser + model.loss_law(th_c)
        ic = model(torch.zeros(1, 1).to(DEVICE)).pow(2).mean()
        return ((model(t_hat) - theta_hat) ** 2).mean(), (res ** 2).mean() + ic

    def total_loss(phase):
        data_loss, physics_loss = losses()
        l = data_weight * data_loss + physics_weight * physics_loss
        l.backward()
        if log is not None:
            log.record(phase, data_loss, physics_loss)
        return l

    cache_path = cache_file("lossnet", t, th, t_off_, adam_epochs, lbfgs_iters, data_weight, physics_weight, seed, n_units)
    if USE_CACHE and log is None and os.path.exists(cache_path):
        model.load_state_dict(torch.load(cache_path, map_location=DEVICE))      # trained before: no training
    else:
        if log is not None:
            log.start(model, T_scale / t_scale)
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)
        for _ in range(adam_epochs):
            opt.zero_grad(); total_loss("Adam"); opt.step()
        if log is not None:
            log.end_adam()
        opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters, line_search_fn="strong_wolfe")
        def closure():
            opt2.zero_grad(); return total_loss("L-BFGS")
        opt2.step(closure)
        if log is not None:
            log.finish(opt2)
        if USE_CACHE and log is None:
            torch.save(model.state_dict(), cache_path)

    a_est = model.a.item() * T_scale / t_scale
    f_est = lambda x: model.loss_law(np_to_th(np.atleast_1d(x) / T_scale)).detach().cpu().numpy().ravel() * T_scale / t_scale
    fit = model(t_hat).detach().cpu().numpy().ravel() * T_scale
    if return_terms:   # the two losses at the end of training (before the weights)
        data_loss, physics_loss = losses()
        return a_est, f_est, fit, dict(data=data_loss.item(), physics=physics_loss.item())
    return a_est, f_est, fit


def linear_layers(model):
    # the Linear layers in forward order: temperature network T1...T4, Tout, then loss network f1, f2, fout
    named = [(name, m) for name, m in model.named_modules() if isinstance(m, nn.Linear)]
    temp = [m for name, m in named if not name.startswith("loss_law")]
    loss = [m for name, m in named if name.startswith("loss_law")]
    names = [f"T{i + 1}" for i in range(len(temp) - 1)] + ["Tout"]
    if loss:
        names += [f"f{i + 1}" for i in range(len(loss) - 1)] + ["fout"]
    return list(zip(names, temp + loss))


class TrainLog:
    # records the losses, a and the gradient of every layer (every 10th Adam step, every L-BFGS evaluation)
    def __init__(self, every=10):
        self.every, self.rows, self.evals = every, [], {"Adam": 0, "L-BFGS": 0}

    def start(self, model, a_scale):
        self.model, self.a_scale, self.layers = model, a_scale, linear_layers(model)

    def record(self, phase, data_loss, physics_loss):
        self.evals[phase] += 1
        if phase == "Adam" and (self.evals[phase] - 1) % self.every != 0:
            return
        row = dict(phase=phase, n_eval=sum(self.evals.values()), data_loss=data_loss.item(),
                   physics_loss=physics_loss.item(), a=self.model.a.item() * self.a_scale)
        for name, m in self.layers:
            row[name] = torch.cat([m.weight.grad.ravel(), m.bias.grad.ravel()]).pow(2).mean().sqrt().item()
        self.rows.append(row)

    def end_adam(self): pass
    def finish(self, lbfgs): pass


def rms(a, b):
    return np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2))


def logged_history(path, adam_lr=None):
    # one logged run of each PINN on DIAG_CURVE (seed 0); read from path if it exists
    if os.path.exists(path):
        return pd.read_csv(path)
    t, T = load_real(REAL_BY_LABEL[DIAG_CURVE])
    Te, toff = REAL_BY_LABEL[DIAG_CURVE]["Tenv"], REAL_BY_LABEL[DIAG_CURVE]["t_off"]
    log1, log2 = TrainLog(), TrainLog()
    original = torch.optim.Adam
    if adam_lr:
        torch.optim.Adam = lambda params, lr=None, **kw: original(params, lr=adam_lr, **kw)
    try:
        with redirect_stdout(io.StringIO()):
            train_photothermal(t, T, toff, Tenv_=Te, seed=0, log=log1)
        train_loss_pinn(t, T - Te, toff, seed=0, log=log2)
    finally:
        torch.optim.Adam = original
    hist = pd.concat([pd.DataFrame(log1.rows).assign(model="linear-loss"), pd.DataFrame(log2.rows).assign(model="learned-loss")], ignore_index=True)
    hist.to_csv(path, index=False)
    return hist

In [ ]:
def roper(t, T, t_off_, Tenv_, min_frac=0.05):
    theta = T - Tenv_
    dT_max = theta[t <= t_off_].max()
    cool = (t > t_off_) & (theta > min_frac * dT_max)
    slope, _ = np.polyfit(t[cool] - t_off_, np.log(theta[cool]), 1)
    tau = -1.0 / slope
    return dT_max / tau, tau


def r2(y, y_hat):                                   # R²: 1 is a perfect fit
    y, y_hat = np.asarray(y), np.asarray(y_hat)
    return 1 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)


def law_curve(t, a, k, t_off_):                     # temperature rise θ(t) for the linear law f = kθ
    return heating_cooling(t, a, 1 / k, t_off_, 0.0)


def learned_curve(t, a, f, t_off_):                 # θ(t) for a learned law f
    return simulate_loss(t, a, lambda x: float(f(np.array([x]))[0]), t_off_)


def split(t, t_off_, share):                        # True = test point
    # the same share of the heating points and of the cooling points, evenly spaced (not random);
    # the first and the last point of each phase stay in the training data
    test = np.zeros(len(t), dtype=bool)
    for phase in (t < t_off_, t >= t_off_):
        idx = np.where(phase)[0]
        n_test = int(round(share * len(idx)))
        test[idx[np.linspace(0, len(idx) - 1, n_test + 2)[1:-1].round().astype(int)]] = True
    return test


def fit_methods(t, T, t_off_, Te, train, seed=0, **kw):
    # fits the four methods on the points where train is True; returns {method: function t -> θ(t)}
    t_tr, T_tr = t[train], T[train]
    th_tr = T_tr - Te
    a_r, tau_r = roper(t_tr, T_tr, t_off_, Te)
    (a_s, k_s), _ = single_tau_fit(t_tr, th_tr, t_off_)
    with redirect_stdout(io.StringIO()):
        a_l, tau_l = train_photothermal(t_tr, T_tr, t_off_, Tenv_=Te, seed=seed, **kw)
    a_n, f_n, _ = train_loss_pinn(t_tr, th_tr, t_off_, seed=seed, **kw)
    return {"Roper": lambda tt: law_curve(tt, a_r, 1 / tau_r, t_off_),
            "Least squares": lambda tt: law_curve(tt, a_s, k_s, t_off_),
            "Linear-loss PINN": lambda tt: law_curve(tt, a_l, 1 / tau_l, t_off_),
            "Learned-loss PINN": lambda tt: learned_curve(tt, a_n, f_n, t_off_)}


models = ["linear-loss", "learned-loss"]
PINNS = ["Linear-loss PINN", "Learned-loss PINN"]
CURVES = ["DS1", "DS2b", "DS4"]

## 1 · Learning rate and epochs

In [ ]:
LRS = [3e-3, 1e-3, 1e-4, 1e-5]
EPOCHS = 16000


def adam_run(lr_):
    # one logged Adam run of each PINN on DIAG_CURVE; read from results/ if it exists
    path = f"results/rev2_adam_lr_{lr_:g}.csv"
    if os.path.exists(path):
        return pd.read_csv(path)
    ds = REAL_BY_LABEL[DIAG_CURVE]
    t, T = load_real(ds)
    Te, toff = ds["Tenv"], ds["t_off"]
    log1, log2 = TrainLog(), TrainLog()
    original = torch.optim.Adam
    torch.optim.Adam = lambda params, lr=None, **kw: original(params, lr=lr_, **kw)
    try:
        with redirect_stdout(io.StringIO()):
            train_photothermal(t, T, toff, Tenv_=Te, seed=0, adam_epochs=EPOCHS, lbfgs_iters=1, log=log1)
        train_loss_pinn(t, T - Te, toff, seed=0, adam_epochs=EPOCHS, lbfgs_iters=1, log=log2)
    finally:
        torch.optim.Adam = original
    hist = pd.concat([pd.DataFrame(log1.rows).assign(model=models[0]), pd.DataFrame(log2.rows).assign(model=models[1])], ignore_index=True)
    hist.to_csv(path, index=False)
    return hist


fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
rows = []
for lr_ in LRS:
    hist = adam_run(lr_)
    for a, m in zip(ax, models):
        h = hist[(hist.model == m) & (hist.phase == "Adam")]
        total = (h.data_loss + h.physics_loss).values
        a.semilogy(h.n_eval, total, lw=1.2, label=f"learning rate {lr_:g}")
        at = lambda e: total[min(np.searchsorted(h.n_eval.values, e), len(total) - 1)]
        rows.append(dict(model=m, learning_rate=lr_, jumps_above_2x=int((total[1:] / total[:-1] > 2).sum()),
                         loss_at_1000=at(1000), loss_at_4000=at(4000), loss_at_16000=at(EPOCHS)))
for a, m in zip(ax, models):
    a.set_title(m); a.set_xlabel("training steps (Adam epochs)"); a.legend(frameon=False, fontsize=8); a.grid(alpha=0.3)
ax[0].set_ylabel("total loss (log scale)")
plt.savefig("results/plots/rev2_1_learning_rate.jpg", dpi=150, bbox_inches="tight"); plt.show()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3g}"))

## 2 · Budget and R²

In [ ]:
BUDGETS = [(1000, 100), (3000, 300), (8000, 1000), (16000, 2000)]
names = [f"{a}+{b}" for a, b in BUDGETS]
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for label, color in zip(CURVES, ["C0", "C1", "C2"]):
    ds = REAL_BY_LABEL[label]
    t, T = load_real(ds)
    Te, toff = ds["Tenv"], ds["t_off"]
    th = T - Te
    test = split(t, toff, 0.20)                                              # 80 % training, 20 % test
    score = {m: ([], []) for m in PINNS}
    for adam, lbfgs in BUDGETS:
        curves = fit_methods(t, T, toff, Te, ~test, adam_epochs=adam, lbfgs_iters=lbfgs)
        for m in PINNS:
            full = curves[m](t)
            score[m][0].append(r2(th[~test], full[~test]))
            score[m][1].append(r2(th[test], full[test]))
    for a, m in zip(ax, PINNS):
        a.plot(names, score[m][1], "o-", color=color, label=f"{label} test")
        a.plot(names, score[m][0], "o--", color=color, alpha=0.5, label=f"{label} training")
for a, m in zip(ax, PINNS):
    a.set_title(m); a.set_xlabel("budget (Adam epochs + L-BFGS iterations)"); a.legend(frameon=False, fontsize=7); a.grid(alpha=0.3)
ax[0].set_ylabel("R²")
plt.savefig("results/plots/rev2_2_budget_r2.jpg", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
rows = []
for label, ds in REAL_BY_LABEL.items():
    t, T = load_real(ds)
    Te, toff = ds["Tenv"], ds["t_off"]
    th = T - Te
    test = split(t, toff, 0.20)
    curves = fit_methods(t, T, toff, Te, ~test)
    rows.append(dict(curve=label, **{m: r2(th[test], curves[m](t)[test]) for m in curves}))
print("R² on the test points (20 % of the points, not used for fitting)")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 3 · a and k of each method

In [ ]:
def recover_k(t, th, toff, seed):
    a, f, _ = train_loss_pinn(t, th, toff, seed=seed)
    x = np.linspace(0.1 * th.max(), th.max(), 50)
    return a, x @ f(x) / (x @ x)                          # least-squares fit of f = kθ to the learned f


def estimate_all(t, T, toff, Te, seed):                   # (a, k) of the four methods, fitted on all points
    th = T - Te
    a_r, tau_r = roper(t, T, toff, Te)
    (a_s, k_s), _ = single_tau_fit(t, th, toff)
    with redirect_stdout(io.StringIO()):
        a_l, tau_l = train_photothermal(t, T, toff, Tenv_=Te, seed=seed)
    return {"Roper": (a_r, 1 / tau_r), "Least squares": (a_s, k_s), "Linear-loss PINN": (a_l, 1 / tau_l),
            "Learned-loss PINN": recover_k(t, th, toff, seed)}


cases = [(name, [make_experiment(**e, seed=s) for s in range(3)], e["t_off_e"], Tenv_true) for name, e in EXPERIMENTS.items()]
cases += [(label, [load_real(ds)] * 3, ds["t_off"], ds["Tenv"]) for label, ds in REAL_BY_LABEL.items()]
rows, params = [], {}
for name, datasets, toff, Te in cases:
    if name in EXPERIMENTS:
        rows.append(dict(data=name, method="True value", a=a_true, k=1 / tau_true))
    runs = [estimate_all(t, T, toff, Te, s) for s, (t, T) in enumerate(datasets)]
    for m in runs[0]:
        a, k = np.mean([r[m] for r in runs], axis=0)                   # mean of three seeds
        params[(name, m)] = (a, k)
        rows.append(dict(data=name, method=m, a=a, k=k))
print("a (°C/s) and k (1/s); synthetic data: true value given; PINNs: mean of three seeds")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

## 4 · Curves rebuilt from a and k of each method

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(9, 10))
for a, (label, ds) in zip(ax.ravel(), REAL_BY_LABEL.items()):
    t, T = load_real(ds)
    th = T - ds["Tenv"]
    a.plot(t / 60, th, color="0.75", lw=3, label="data")
    for m, style in zip(["Roper", "Least squares", "Linear-loss PINN", "Learned-loss PINN"], ["C3--", "C0-", "C2:", "C1-."]):
        y = law_curve(t, *params[(label, m)], ds["t_off"])
        a.plot(t / 60, y, style, lw=1.5, label=f"{m} (R² {r2(th, y):.3f})")
    a.set_title(label); a.set_xlabel("Time (min)"); a.legend(fontsize=6.5, loc="lower center")
ax[2, 1].axis("off")
fig.subplots_adjust(hspace=0.5, wspace=0.25)
plt.savefig("results/plots/rev2_4_curves_each_method.jpg", dpi=150, bbox_inches="tight"); plt.show()

## 5 · Train/test portion

In [ ]:
SHARES = [0.10, 0.15, 0.20, 0.25, 0.30]     # test share; the rest is training (90, 85, 80, 75, 70 %)
rows = []
for label in CURVES:
    ds = REAL_BY_LABEL[label]
    t, T = load_real(ds)
    Te, toff = ds["Tenv"], ds["t_off"]
    th = T - Te
    for share in SHARES:
        test = split(t, toff, share)
        curves = fit_methods(t, T, toff, Te, ~test)
        rows.append(dict(curve=label, train=f"{100 * (1 - test.mean()):.0f} %", test=f"{100 * test.mean():.0f} %",
                         train_heating=int((t[~test] < toff).sum()), train_cooling=int((t[~test] >= toff).sum()),
                         test_heating=int((t[test] < toff).sum()), test_cooling=int((t[test] >= toff).sum()),
                         **{m: r2(th[test], curves[m](t)[test]) for m in curves}))
print("R² on the test points; heating = before the laser is switched off, cooling = after")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 6 · Physics and data weights

In [ ]:
WEIGHTS = [0.01, 0.1, 1.0, 10.0, 100.0]       # physics weight, the data weight is 1
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
rows = []
for label, color in zip(CURVES, ["C0", "C1", "C2"]):
    ds = REAL_BY_LABEL[label]
    t, T = load_real(ds)
    Te, toff = ds["Tenv"], ds["t_off"]
    th = T - Te
    test = split(t, toff, 0.20)
    score = {m: [] for m in PINNS}
    for w in WEIGHTS:
        curves = fit_methods(t, T, toff, Te, ~test, data_weight=1.0, physics_weight=w)
        for m in PINNS:
            score[m].append(r2(th[test], curves[m](t)[test]))
    for a, m in zip(ax, PINNS):
        a.semilogx(WEIGHTS, score[m], "o-", color=color, label=label)
    rows += [dict(curve=label, model=m, **dict(zip(WEIGHTS, score[m]))) for m in PINNS]
for a, m in zip(ax, PINNS):
    a.set_title(m); a.set_xlabel("physics weight (data weight = 1)"); a.legend(frameon=False); a.grid(alpha=0.3)
ax[0].set_ylabel("R² on the test points")
plt.savefig("results/plots/rev2_6_weights.jpg", dpi=150, bbox_inches="tight"); plt.show()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 7 · Three curves for training, two for evaluation

In [ ]:
TRAIN, EVAL = ["DS1", "DS2a", "DS2b"], ["DS3", "DS4"]       # split by source paper
laws = {}
for label in TRAIN:
    ds = REAL_BY_LABEL[label]
    t, T = load_real(ds)
    laws[label] = train_loss_pinn(t, T - ds["Tenv"], ds["t_off"], seed=0)[1]      # the law f learned on this curve
rows = []
for label in EVAL:
    ds = REAL_BY_LABEL[label]
    t, T = load_real(ds)
    toff, th = ds["t_off"], T - ds["Tenv"]
    a_own, f_own, _ = train_loss_pinn(t, th, toff, seed=0)
    (a_s, k_s), _ = single_tau_fit(t, th, toff)
    row = dict(curve=label, own_law=r2(th, learned_curve(t, a_own, f_own, toff)), linear_law=r2(th, law_curve(t, a_s, k_s, toff)))
    for name, f in laws.items():                        # a law learned on another curve; only a is fitted here
        res = least_squares(lambda p: learned_curve(t, p[0], f, toff) - th, [0.05], bounds=([1e-5], [5]), diff_step=1e-3, x_scale="jac")
        row[f"law of {name}"] = r2(th, learned_curve(t, res.x[0], f, toff))
    rows.append(row)
print("R² on the whole evaluation curve; the law f is fixed, only a is fitted")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))